In [1]:
import os
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [2]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
from datasets import load_dataset

# Load squadv2 (use 'distractor' for RAG-style evaluation)
dataset = load_dataset("squad_v2", split="validation")

# small subset for testing
dataset = dataset.select(range(100))  # start small

Generating validation split: 100%|██████████| 11873/11873 [00:00<00:00, 42527.82 examples/s]


In [33]:
dataset[5]

{'id': '5ad39d53604f3c001a3fe8d1',
 'title': 'Normans',
 'context': 'The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries.',
 'question': "Who gave their name to Normandy in the 1000's and 1100's",
 'answers': {'text': [], 'answer_start': []}}

In [4]:
from collections import Counter

def get_most_frequent_answer(row):
    ans_list = row['answers']['text']
    
    # Check if the list is empty (unanswerable questions)
    if not ans_list:
        return "N/A"
    
    # Count occurrences of each unique string
    counts = Counter(ans_list)
    
    # .most_common(1) returns a list like [('France', 4)]
    # We take the first element [0] and then the text [0]
    most_common_answer = counts.most_common(1)[0][0]
    
    return most_common_answer

# Example usage:
# row = {'answers': {'text': ['France', 'France', 'France', 'France']}}
# result -> 'France'

In [ ]:
# loader= DirectoryLoader('documents',glob='*.pdf',loader_cls=PyPDFLoader)
# docs=loader.lazy_load()

In [5]:
from langchain_core.documents import Document

documents = []
qa_pairs = []


for item in dataset:
    qa_pairs.append({
        "question": item["question"],
        "answer": get_most_frequent_answer(item)
    })
    context = item["context"]
    # sentences_list = contexts["sentences"]

    # all_sentences = []
    # for sentences in sentences_list:
    #     all_sentences.extend(sentences)

    # full_text = "\n".join(all_sentences)

    documents.append(
        Document(
            page_content=context
        )
    )


print("Total docs:", len(documents))

Total docs: 100


In [6]:
#KG implementation
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class Triple:
    subj: str
    rel: str
    obj: str

class SimpleKG:
    def __init__(self):
        self.triples: List[Triple] = []

    def add_triple(self, subj: str, rel: str, obj: str):
        self.triples.append(Triple(subj, rel, obj))

    def find_triples(self, entity: str) -> List[Triple]:
        # return all triples where entity is subject or object
        return [t for t in self.triples if t.subj == entity or t.obj == entity]


KG = SimpleKG()
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_triples_spacy(text):
    doc = nlp(text)
    triples = []
    for token in doc:
        if token.dep_ in ("ROOT", "relcl"):  # verbs or relations
            subj = [w.text for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w.text for w in token.rights if w.dep_ in ("dobj", "pobj", "attr")]
            if subj and obj:
                triples.append((" ".join(subj), token.lemma_, " ".join(obj)))
    return triples




#link entities in query to KG entities
import spacy
nlp = spacy.load("en_core_web_sm")
def extract_entity_mentions(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents] or [chunk.text for chunk in doc.noun_chunks]

def link_entities(query, kg_entities):
    # simple substring + optional embedding similarity
    mentions = extract_entity_mentions(query)  
    entity_map = {}
    for m in mentions:
        entity_map[m] = [e for e in kg_entities if m.lower() in e.lower()]
    return entity_map

def retrieve_kg_context(query, KG: SimpleKG):
    kg_entities = list(set([t.subj for t in KG.triples] + [t.obj for t in KG.triples]))
    entity_map = link_entities(query, kg_entities)
    triples_text = []
    for ents in entity_map.values():
        for ent in ents:
            for t in KG.find_triples(ent):
                triples_text.append(f"{t.subj} {t.rel} {t.obj}")
    return "\n".join(triples_text)



In [58]:

#combine kg retrieval and context retrieval
# def get_combined_context(query, retriever, KG):
#     # 1. Retrieve text from FAISS DB
#     text_docs = retriever.invoke(query)
#     text_context = "\n\n".join([d.page_content for d in text_docs])
#     #print(f"context:{text_context}")
#     if(len(text_docs)==0):
#         return None
    
#     # 2. Retrieve KG triples
#     kg_context = retrieve_kg_context(query, KG)

#     # 3. Combine for final LLM input
#     combined_context = f"KG Facts:\n{kg_context}\n\nTextual Context:\n{text_context}"
#     return combined_context

def get_combined_context(query, retriever, KG):
    # 1. Text Retrieval & Deduplication
    text_docs = retriever.invoke(query)[:3]
    
    
    unique_texts = list(dict.fromkeys([d.page_content.strip() for d in text_docs]))
    clean_text = "\n\n".join(unique_texts)

    # 2. KG Retrieval & Deduplication
    kg_raw = retrieve_kg_context(query, KG)[:100] # Assuming this returns a string or list
    
    
    # If it's a string with newlines:
    if isinstance(kg_raw, str):
        kg_lines = [line.strip() for line in kg_raw.split('\n') if line.strip()]
        unique_kg = list(dict.fromkeys(kg_lines))
        clean_kg = "\n".join(unique_kg)
    # If it's already a list:
    else:
        unique_kg = list(dict.fromkeys([str(fact).strip() for fact in kg_raw]))
        clean_kg = "\n".join(unique_kg)

    return f"KG Facts:\n{clean_kg}\n\nTextual Context:\n{clean_text}"

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", " ", ""]
)

chunks= text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

for doc in documents:
    triples = extract_triples_spacy(doc.page_content)
    for (subj, rel, obj) in triples:
        KG.add_triple(subj.strip(), rel.strip(), obj.strip())




Total chunks: 129


In [9]:
print(chunks[100].page_content[:])

A few years after the First Crusade, in 1107, the Normans under the command of Bohemond, Robert's son, landed in Valona and besieged Dyrrachium using the most sophisticated military equipment of the time, but to no avail. Meanwhile, they occupied Petrela, the citadel of Mili at the banks of the river Deabolis, Gllavenica (Ballsh), Kanina and Jericho. This time, the Albanians sided with the Normans, dissatisfied by the heavy taxes the Byzantines had imposed upon them. With their help, the Normans secured the Arbanon passes and opened their way to Dibra. The lack of supplies, disease and Byzantine resistance forced Bohemond to retreat from his campaign and sign a peace treaty with the Byzantines in the city of Deabolis.


In [10]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": "cuda:2"},
                                   encode_kwargs={"normalize_embeddings": True}
                                   )

/tmp/ipykernel_1099225/1272004119.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [11]:
#FAISS cosine similarity index
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

dimension = 384


index = faiss.IndexFlatIP(dimension)

In [12]:

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
print(len(chunks))
print(vectorstore.index.ntotal)

129
129


In [13]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
    )

In [14]:
llm = HuggingFacePipeline.from_model_id(
    # model_id="/mnt/nas/shuvranshu/huggingface_cache/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", 
    model_id="meta-llama/Llama-3.1-8B",
    # model_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "do_sample": False,
    },
    device=1
)


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.38it/s]


In [67]:


actor_prompt = PromptTemplate(
    input_variables=["context", "question","feedback_section"],
    template="""
You are a factual assistant whose goal is to produce answers strictly supported by the provided context.



Instructions:
1. Use ONLY the information present in the provided context.
2. Do NOT introduce external knowledge.
3. If the answer cannot be found in the context, respond with: NO.
4. Ensure the answer directly addresses the question.
5. Answer in maximum 3 to 5 words
6. If the answer is implied, infer it.


Context:
{context}

Question:
{question}

Refined Answer:
    """
)

def actor(question, context, feedback_section):
    actor_chain = actor_prompt | llm
    # context = get_combined_context(question,retriever, KG)
    response = actor_chain.invoke({
            "context":context,
            "feedback_section": feedback_section,
            "question":question
            }
            )
    answer=response.split('Answer:')[-1].strip()
    return answer


In [20]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a factual assistant. Use the following context to answer the question.
Do NOT add information that is not supported by the context.
if you don't find the answer in the context then simply say NO.

Context:
{context}

Question: {question}
Answer:
"""
)

In [21]:
question="summarize this document"
context = get_combined_context(question,retriever, KG)
chain = prompt | llm
response = chain.invoke({
        "context":context,
        "question":question,
        }
        )
answer=response.split('Answer:')[-1].strip()
print(f"answer:{answer}")    


/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


answer:The descendants of Rollo's Vikings and their Frankish wives would replace the Norse religion and Old Norse language with Catholicism (Christianity) and the Gallo-Romance language of the local people, blending their maternal Frankish heritage with Old Norse traditions and customs to synthesize a unique "Norman" culture in the north of France. The Norman language was forged by the adoption of the indigenous langue d'oïl branch of Romance by a Norse-speaking ruling class, and it developed into the regional language that survives today.


In [16]:
#evaluator
import json
from langchain_community.chat_models import ChatOllama


In [55]:
evaluator_llm= ChatOllama(model="qwen2.5:7b",
                          format="json",
                          temperature=0)
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a RAG Quality Auditor. You must evaluate ONLY two things:
    1. CONTEXT RELEVANCE: Does the retrieved context contain the information needed to answer the user's query?
    2. FAITHFULNESS: Is the provided answer derived ONLY from the retrieved context?
     
    Output ONLY a JSON object with EXACTLY this structure:
    {{
        "context_relevance": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "faithfulness": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "final_status": "PASS" or "FAIL",
        "action_item": "What should the system do next?(e.g., 'Re-retrieve context' or 'Refine answer').
        if context_relevance score<0.8 then 'Re-retrieve context',if faithfullness score<0.8 then 'Refine answer'.
    }}
    
    Threshold for PASS: Both scores must be >= 0.8.
    """),
    ("user", "USER QUERY: {query}\n\nRETRIEVED CONTEXT: {context}\n\nGENERATED ANSWER: {answer}")
])

In [18]:
def evaluate_response(question,context, answer):
    chain = eval_prompt | evaluator_llm
    # response = chain.invoke({"context": context, "answer": answer})
    
    
    try:
        response = chain.invoke({
            "query": question,
            "context": context,
            "answer": answer
        })
        return json.loads(response.content)
    except Exception as e:
        return {
            "final_status": "FAIL",
            "action_item": f"Error in evaluation: {str(e)}"
        }

In [19]:

try:
    response = evaluator_llm.invoke("Hello!")
    print("Ollama is awake and responding!")
except Exception as e:
    print(f"Connection failed. Is the server running? Error: {e}")

Ollama is awake and responding!


In [39]:
query_refiner_llm = ChatOllama(model="qwen2.5:7b", temperature=0)

def generate_refined_query(original_query, critique):
    prompt = f"""
    The initial search for '{original_query}' failed.
    Critique: {critique}
    based on this critique, write a better search query which contains the missing terms or concepts that the original query lacked,
    which led to the failure.
    Don't change the meaning of the question.
    Output ONLY the new search string.
    """
    response = query_refiner_llm.invoke(prompt)
    return response.content.strip()

In [70]:
def run_reflexion_loop(original_question,max_iterations=3):
    
    search_query = original_question
    current_context = get_combined_context(search_query, retriever, KG)
    feedback_section = " " 
    
    for i in range(max_iterations):
        # print(f"\n--- Iteration {i+1} ---")
        
        answer = actor(original_question, current_context, feedback_section)
        
        evaluation = evaluate_response(original_question, current_context, answer)
        # print(json.dumps(evaluation, indent=2))
        # print(current_context)
        # print(f"answer:{answer}")

        if evaluation.get('final_status') == "PASS":
            return answer,current_context
        
        action = evaluation.get('action_item', 'Refine answer')
        
        if action == "Re-retrieve context":
            search_query = generate_refined_query(original_question, evaluation['context_relevance']['reason'])
            # print(f"new search query:{search_query}")
            current_context = get_combined_context(search_query, retriever, KG)
        else:
            feedback_section=evaluation['faithfulness']['reason']
        # feedback_section = f"""
        # ### FEEDBACK FROM PREVIOUS ATTEMPT ###
        # - Context Relevance Score: {evaluation['context_relevance']['score']}
        # - Context Critique: {evaluation['context_relevance']['reason']}
        
        # - Faithfulness Score: {evaluation['faithfulness']['score']}
        # - Faithfulness Critique: {evaluation['faithfulness']['reason']}
        
        # - Required Action: {evaluation['action_item']}
        # ######################################
        # """
        
        # print(f"Critique sent to Actor: {evaluation['action_item']}")
        
    # print(" Max iterations reached.")
    return answer,current_context

In [39]:
dataset[1]["context"]

'The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries.'

In [68]:
question=qa_pairs[1]["question"]
print(question)
answer,context=run_reflexion_loop(question)
print(context)
print(f"answer:{answer}")
print(qa_pairs[1]["answer"])

When were the Normans in Normandy?

--- Iteration 1 ---
{
  "context_relevance": {
    "score": 0.8,
    "reason": "The retrieved context provides information about when the Normans emerged as a distinct cultural identity, which is relevant to the user's query."
  },
  "faithfulness": {
    "score": 1.0,
    "reason": "The generated answer is directly derived from the provided context."
  },
  "final_status": "PASS",
  "action_item": "No action needed."
}
KG Facts:
Normans adopt doctrines
Cris

Textual Context:
The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their

In [66]:
my_prompt = PromptTemplate(
    input_variables=[ "question"],
    template="""
You are a factual assistant whose goal is to produce answers strictly supported by the provided context.

Instructions:
1. Use ONLY the information present in the provided context.
2. Do NOT introduce external knowledge.
3. If the answer cannot be found in the context, respond with: NO.
4. Ensure the answer directly addresses the question.
5. Answer in maximum 3 to 5 words
6. If the answer is implied, infer it.


Context:
KG Facts:
Normans adopt doctrines
Cris

Textual Context:
The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name 
to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, 
Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations 
of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the 
Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first 
half of the 10th century, and it continued to evolve over the succeeding centuries.

Question:
{question}

Refined Answer:
    """
)

my_chain=my_prompt|llm

answer=my_chain.invoke("When were the Normans in Normandy?")
print(answer)

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



You are a factual assistant whose goal is to produce answers strictly supported by the provided context.

Instructions:
1. Use ONLY the information present in the provided context.
2. Do NOT introduce external knowledge.
3. If the answer cannot be found in the context, respond with: NO.
4. Ensure the answer directly addresses the question.
5. Answer in maximum 3 to 5 words
6. If the answer is implied, infer it.


Context:
KG Facts:
Normans adopt doctrines
Cris

Textual Context:
The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name 
to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, 
Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations 
of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually m

In [71]:
# question="Explain the legal basis and the override power of the inserted rule 31D."
# answer=run_reflexion_loop(question)
# print(f"answer:{answer}")

results = []

for i, qa in enumerate(qa_pairs):
    question = qa["question"]
    gt = qa["answer"]
    response,context = run_reflexion_loop(question)
    
    results.append({
        "question": question,
        "pred": response,
        "gt": gt,
        "context":context
    })

    print(f"q{i}:{response}")
    # print(f"Done {i}")

q0:France
q1:10th and 11th centuries
q2:Danes
q3:Rollo
q4:10th century
q5:Normans
q6:France is a region of Normandy.
q7:King Charles III swore fealty to King Edward the Confessor.
q8:The Frankish identity emerged in the first half of the 10th century.
q9:Duke William II of Normandy
q10:Rollo
q11:Christianity
q12:political, cultural and military impact
q13:The Maniakates were famed for their Christian spirit.
q14:The Normans assimilated the Roman language.
q15:Rollo
q16:Principality of Antioch
q17:Norseman, Viking
q18:9th century
q19:Normans/Normanz
q20:10th century
q21:911
q22:King Charles III of West Francia
q23:Epte
q24:1085
q25:Treaty of Saint-Clair-sur-Epte
q26:Rollo
q27:The French promised to protect Rollo and his men from the Bretons.
q28:Normans
q29:880s
q30:Rollo conquered the Vikings in the Cotentin Peninsula and the Pays de Caux.
q31:Christianity
q32:Normandy
q33:Catholicism (Christianity)
q34:Old Norse traditions merged with the Gallo-Romance language of the local people.
q3

In [72]:
#exact match and F1 score
import re
import string

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [73]:
def compute_em(pred, gt):
    return int(normalize_answer(pred) == normalize_answer(gt))

In [74]:
def compute_f1(pred, gt):
    pred_tokens = normalize_answer(pred).split()
    gt_tokens = normalize_answer(gt).split()

    common = set(pred_tokens) & set(gt_tokens)
    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)

In [76]:
em_scores = []
f1_scores = []

for r in results:
    pred = r["pred"]
    gt = r["gt"]

    if pred is None or pred.strip() == "":
        continue   # skip bad outputs

    em_scores.append(compute_em(pred, gt))
    f1_scores.append(compute_f1(pred, gt))

avg_em = sum(em_scores) / len(em_scores)
avg_f1 = sum(f1_scores) / len(f1_scores)

print("Exact Match (EM):", avg_em)
print("F1 Score:", avg_f1)

Exact Match (EM): 0.24210526315789474
F1 Score: 0.28568922305764416


In [77]:

#ragas evaluation
dataset = []

for item in results:
    dataset.append({
        "user_input": item["question"],
        "retrieved_contexts": item["context"] if isinstance(item["context"], list) else [item["context"]],
        "response": item["pred"],
        "reference": item["gt"]
    })

In [78]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)

In [79]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from langchain_community.embeddings import OllamaEmbeddings
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

In [80]:
json_prompt = """
You are a strict JSON output generator used for evaluation in RAG systems.
You must always reply ONLY with a valid JSON object — no explanations, no extra text.

If the question asks for a score, output: {"score": <float between 0 and 1>}
If the question asks for a label (e.g., yes/no), output: {"label": "yes"} or {"label": "no"}
Do not include extra keys, comments, or markdown fences.
"""
langchain_llm= ChatOllama(model="mistral",temperature=0,system=json_prompt)


langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")

/tmp/ipykernel_1099225/241535895.py:12: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")


In [81]:
result = evaluate(dataset=evaluation_dataset,
                  batch_size=2,
                  metrics=[ LLMContextRecall(), Faithfulness(),FactualCorrectness()],
                  llm=langchain_llm,embeddings=langchain_embeddings)
print(result)

Evaluating:  25%|██▍       | 74/300 [04:14<11:39,  3.09s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[75]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating:  27%|██▋       | 82/300 [04:56<15:02,  4.14s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output pa

{'context_recall': 0.5083, 'faithfulness': 0.5748, 'factual_correctness(mode=f1)': 0.4558}
